In [ ]:
API_URL = "https://api.tip-master-zmt.click/"
API_KEY = "xxxxxxxxxxxxx"
API_SECRET = "xxxxxxxxxxxxxxx"

In [30]:
import os
os.environ["OSPARC_DEV_FEATURES_ENABLED"] = "1"

In [31]:
from multiprocessing.pool import AsyncResult
from tenacity import AsyncRetrying, TryAgain, retry_if_exception_type
from tenacity.wait import wait_fixed
from tqdm.auto import tqdm
from rich import print
import asyncio
import functools
import osparc
import typing


cfg = osparc.Configuration(
    host=f"{API_URL}",
    username=API_KEY,
    password=API_SECRET,
)
# cfg.debug = True

In [32]:
with osparc.ApiClient(cfg) as api_client:
    profile = osparc.UsersApi(api_client).get_my_profile()
    print(profile)
    solvers_api = osparc.SolversApi(api_client)
    files_api = osparc.FilesApi(api_client)


{'first_name': None,
 'gravatar_id': None,
 'groups': {'all': {'description': 'all users',
                    'gid': '1',
                    'label': 'Everyone'},
            'me': {'description': 'primary group',
                   'gid': '26',
                   'label': 'anderegg'},
            'organizations': [{'description': '',
                               'gid': '64',
                               'label': 'osparc devs'}]},
 'id': 23,
 'last_name': None,
 'login': 'anderegg@itis.swiss',
 'role': 'ADMIN'}

In [33]:
elec_round_file = files_api.upload_file("/home/sanderegg/Downloads/Elec_round F 1.yaml")
print(elec_round_file)

{'checksum': '2591bc083bee00f805afb5cd9b9525e387e4cdf5ea998c72a0ec735e9bce33b1',
 'content_type': 'application/octet-stream',
 'e_tag': '02bc5acc1d02f48dec27cf67767d244a',
 'filename': 'Elec_round F 1.yaml',
 'id': 'fa3a4251-3e4a-3916-9797-23299561cdfb'}

In [34]:
runner_content_code_zip_file = files_api.upload_file("/home/sanderegg/Downloads/runner_content_code.zip")
print(runner_content_code_zip_file)

{'checksum': '80b6cf1f613e29fb72dc5174a81d5616e75e0a8141eed04ebaee1e441f653b3d',
 'content_type': 'application/zip',
 'e_tag': '1fc9411e39075896ec8b968d1130e405-11',
 'filename': 'runner_content_code.zip',
 'id': 'ffc5f892-21e8-373e-8d4e-469703e067b1'}

In [35]:
solver = solvers_api.get_solver_release("simcore/services/comp/s4l-python-runner", "1.2.236")
print(solver)

{'description': 
'https://raw.githubusercontent.com/ZurichMedTech/s4l-assets/refs/heads/main/app/full/services/python-runners.md',
 'id': 'simcore/services/comp/s4l-python-runner',
 'maintainer': 'guidon@itis.swiss',
 'title': 'Sim4Life Python Runner',
 'url': 'http://api.tip-master-zmt.click/v0/solvers/simcore/services/comp/s4l-python-runner/releases/1.2.236',
 'version': '1.2.236'}

In [37]:
job = solvers_api.create_job(
    solver_key=solver.id,
    version=solver.version,
    job_inputs=osparc.JobInputs({"input_1": elec_round_file, "input_2": runner_content_code_zip_file}),)
print(job)

{'created_at': datetime.datetime(2026, 3, 17, 15, 28, 53, 571000, tzinfo=tzutc()),
 'id': '15bd790a-e3e7-4481-b4e0-b96ae1f3cf09',
 'inputs_checksum': '33b8f8088fe84f5b64c5bbcd1972e5220a3d31bff4df3712ad4e0d57324c7735',
 'name': 
'solvers/simcore%2Fservices%2Fcomp%2Fs4l-python-runner/releases/1.2.236/jobs/15bd790a-e3e7-4481-b4e0-b96ae1f3cf09',
 'outputs_url': 
'http://api.tip-master-zmt.click/v0/solvers/simcore/services/comp/s4l-python-runner/releases/1.2.236/jobs/15bd790a-
e3e7-4481-b4e0-b96ae1f3cf09/outputs',
 'runner_name': 'solvers/simcore%2Fservices%2Fcomp%2Fs4l-python-runner/releases/1.2.236',
 'runner_url': 
'http://api.tip-master-zmt.click/v0/solvers/simcore/services/comp/s4l-python-runner/releases/1.2.236',
 'url': 
'http://api.tip-master-zmt.click/v0/solvers/simcore/services/comp/s4l-python-runner/releases/1.2.236/jobs/15bd790a-
e3e7-4481-b4e0-b96ae1f3cf09'}

In [38]:
status = solvers_api.start_job(solver.id, solver.version, job.id)
print(status)

{'job_id': '15bd790a-e3e7-4481-b4e0-b96ae1f3cf09',
 'progress': 0,
 'started_at': None,
 'state': 'PUBLISHED',
 'stopped_at': None,
 'submitted_at': datetime.datetime(2026, 3, 17, 15, 29, 2, 52203, tzinfo=tzutc())}

In [41]:
solvers_api.inspect_job(solver.id, solver.version, job.id)

{'job_id': '15bd790a-e3e7-4481-b4e0-b96ae1f3cf09',
 'progress': 100,
 'started_at': datetime.datetime(2026, 3, 17, 15, 33, 19, 849916, tzinfo=tzutc()),
 'state': 'FAILED',
 'stopped_at': datetime.datetime(2026, 3, 17, 15, 51, 39, 943769, tzinfo=tzutc()),
 'submitted_at': datetime.datetime(2026, 3, 17, 15, 29, 2, 52203, tzinfo=tzutc())}

In [42]:
solvers_api.get_job_output_logfile(solver.id, solver.version, job.id)

'/tmp/tmp9xgwy7g4'